In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
from tqdm import tqdm
from torch import nn, optim
import torchvision.utils as vutils

# Set random seed and device (CPU or GPU)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [2]:

class YelpImageDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform
        self.df['label_code'] = self.df['label'].astype('category').cat.codes

        # Use only images that exist in expected label subfolder
        def get_full_path(row):
            img_name = os.path.basename(row['image_path'])
            label_folder = str(row['label'])
            path = os.path.join(image_dir, label_folder, img_name)
            return path if os.path.exists(path) else None
        self.df['full_path'] = self.df.apply(get_full_path, axis=1)
        self.df = self.df[self.df['full_path'].notnull()]
        self.df.reset_index(drop=True, inplace=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['full_path']).convert("RGB")
        label = int(row['label_code'])
        if self.transform:
            image = self.transform(image)
        return image, label


In [3]:

csv_path = '../PREPROCESSED/processed/train_meta.csv'
image_dir = '../PREPROCESSED/processed/train'
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset_full = YelpImageDataset(csv_path, image_dir, transform)
print("Total valid images found:", len(dataset_full))

subset_size = 1000
dataset = Subset(dataset_full, range(subset_size))
print("Subset size for training:", len(dataset))


Total valid images found: 159996
Subset size for training: 1000


In [4]:
# Step 4: Class imbalance sampler
def get_balanced_sampler(dataset):
    if len(dataset) == 0:
        return None
    labels = [dataset[i][1] for i in range(len(dataset))]
    class_sample_counts = np.bincount(labels)
    weights = 1. / class_sample_counts[labels]
    return WeightedRandomSampler(weights, len(weights))


In [5]:

class Generator(nn.Module):
    def __init__(self, nz):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(nz, 256), nn.ReLU(True),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.ReLU(True),
            nn.Linear(512, 3*64*64), nn.Tanh()
        )
    def forward(self, z):
        return self.main(z).view(z.size(0), 3, 64, 64)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3*64*64, 512), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        return self.main(x)


In [6]:

def train_wgan_one_config(hparams):
    lr = hparams['lr']
    latent_dim = hparams['latent_dim']
    batch_size = hparams['batch_size']
    critic_iters = hparams['critic_iters']
    clip_value = hparams['clip_value']
    epochs = hparams['epochs']

    # Split into train and ignore val for data leakage prevention
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, _ = torch.utils.data.random_split(dataset, [train_size, val_size])
    sampler = get_balanced_sampler(train_dataset)
    dataloader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)

    G = Generator(latent_dim).to(device)
    D = Discriminator().to(device)
    optimizer_G = optim.RMSprop(G.parameters(), lr=lr)
    optimizer_D = optim.RMSprop(D.parameters(), lr=lr)

    for epoch in range(epochs):
        loop = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for imgs, _ in loop:
            imgs = imgs.to(device)
            for _ in range(critic_iters):
                z = torch.randn(imgs.size(0), latent_dim).to(device)
                fake_imgs = G(z).detach()
                loss_D = -torch.mean(D(imgs)) + torch.mean(D(fake_imgs))
                optimizer_D.zero_grad()
                loss_D.backward()
                optimizer_D.step()
                for p in D.parameters():
                    p.data.clamp_(-clip_value, clip_value)
            z = torch.randn(imgs.size(0), latent_dim).to(device)
            gen_imgs = G(z)
            loss_G = -torch.mean(D(gen_imgs))
            optimizer_G.zero_grad()
            loss_G.backward()
            optimizer_G.step()
            loop.set_postfix(loss_D=loss_D.item(), loss_G=loss_G.item())
    os.makedirs("wgan_model", exist_ok=True)
    torch.save(G.state_dict(), "wgan_model/wgan_generator.pth")
    print("Model saved: wgan_model/wgan_generator.pth")
    return loss_D.item(), hparams


In [7]:

best_config = {
    'lr': 0.0001,
    'latent_dim': 100,
    'batch_size': 64,
    'critic_iters': 1,
    'clip_value': 0.01,
    'epochs': 30
}
print("Training config:", best_config)
try:
    d_loss, config = train_wgan_one_config(best_config)
    print("Training complete. Final D loss:", d_loss)
except Exception as e:
    print("ERROR:", e)


Training config: {'lr': 0.0001, 'latent_dim': 100, 'batch_size': 64, 'critic_iters': 1, 'clip_value': 0.01, 'epochs': 30}


Model saved: wgan_model/wgan_generator.pth
Training complete. Final D loss: -10.520483016967773


In [8]:

G = Generator(best_config['latent_dim']).to(device)
G.load_state_dict(torch.load("wgan_model/wgan_generator.pth", map_location=device))
G.eval()

os.makedirs("generated_samples", exist_ok=True)
with torch.no_grad():
    z = torch.randn(5, best_config['latent_dim']).to(device)
    fake_imgs = G(z).detach().cpu()
    for i in range(5):
        vutils.save_image(fake_imgs[i], f"generated_samples/sample_{i+1}.png", normalize=True)
print("5 images generated and saved in generated_samples/")


5 images generated and saved in generated_samples/
